In [1]:
%matplotlib widget
# Add the directory containing the package to sys.path
import sys, os
package_dir = os.path.abspath("C:/Users/froll/Documents/Labo/Projets/Outils/swd")
if package_dir not in sys.path:
    sys.path.insert(0, package_dir)
    
from swd import spherical_processing as sp
from swd import geotools as geo
from swd import plots as splots
import swd as swd
import importlib

importlib.reload(swd.plots)
import numpy as np
from numpy import pi, cos, sin
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")
from tqdm import tqdm
np.set_printoptions(precision=2, suppress=True)
# Enable LaTeX rendering
plt.rc('text', usetex=True)
# Improve resolution
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300

# Import utils_DirViolins
scripts_dir = os.path.abspath("C:/Users/froll/Documents/Labo/Projets/Violon/scripts")
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)
import utils_DirViolins as utils
importlib.reload(utils)
from scipy.spatial.transform import Rotation

In [2]:
def get_rotation_matrix(phi, theta):
    """
    Returns a rotation matrix defined by azimuth phi and colatitude theta.
    This rotation aligns the z-axis with the direction given by (phi, theta).
    The rotation is composed of a rotation around y by theta, followed by a rotation around z by phi.
    """
    R_phi = np.array([
        [np.cos(phi), -np.sin(phi), 0],
        [np.sin(phi), np.cos(phi), 0],
        [0, 0, 1]
    ])
    
    R_theta = np.array([
        [np.cos(theta), 0, np.sin(theta)],
        [0, 1, 0],
        [-np.sin(theta), 0, np.cos(theta)]
    ])
    
    # R such that v_rotated = R @ v
    return R_phi @ R_theta

In [3]:
#Vitesse du son au moment de la mesure, dependant de la temperature:
Tc = 21.5 
C = np.sqrt( 1.4 * 287 *(Tc + 273) )
Path = './'
NbMems = 256
NbViolTot = 6
NbViol = 6
OSH = 7
Dyn = 36

NbSH = (OSH+1)**2
NumViolon = np.load('./../results/NumViolon.npz')['NumViolon']
XYZViolins = np.load('./../results/XYZViolinsAligned.npy')

NbMics = 256
NbSrcs = XYZViolins.shape[0] - NbMics

XYZHammerImpact = XYZViolins[NbSrcs-1]
XYZ = XYZViolins-XYZHammerImpact
XYZs = XYZ[:NbSrcs-1]
XYZm = XYZ[NbSrcs:]
XYZHammerImpact = XYZ[NbSrcs-1]

In [4]:
Pm = np.load('./../results/ViolinsFRFsAndRIs.npz')['Hhm']
frq = np.load('./../results/ViolinsFRFsAndRIs.npz')['frq']
frqMax = 12500
CyclicScale = 'icefire' #edge, icefire, phase, hsv
RealScale = 'seismic'
Magnitudescale = 'inferno'

## Process the Spherical Harmonics Spectra for one Violin

In [5]:
Rm = np.linalg.norm(XYZm,axis=1)
Rmin = np.min(Rm)

In [6]:
Band = (frq>0) & (frq<frqMax)
f = frq[Band]
Nbf = len(f)
kvect= 2*pi*f.T/C
Nang = 50
NbTh, NbPh = 2*Nang+2, 2*Nang+2
ThPh, weights = swd.geotools.create_equal_angle_grid(Nang)

NumV = 0
P  = Pm[NumV, Band, :].T
O_SH_vect = sp.compute_N_SH_vect(f,N_SH_max = OSH,rmin = Rmin)


# Mesure de la variance de l'estimation de la décomposition SH
## Principe
Les capteurs restent fixes aux positions $(r_i, \theta_i, \varphi_i)$. 

Le champ $\mathbf{p}$ est mesuré une seule fois à ces positions. 

Ce qui change, c'est la base d'harmoniques sphériques utilisée pour la décomposition : on la tourne par $R(\alpha,\beta,\gamma)$

### Étape 1 — Décomposition de référence (base non tournée)
On construit $\textbf{H}_0 = \mathbf{B}\mathbf{Y}_0$  avec les $b_n(kr_i)$ et $Y_n^m(\theta_i, \varphi_i)$ standards et on calcule :

$$C_0 = \mathbf{H}_0^+ \, \mathbf{p}$$

In [7]:
# Estimation du diagramme de référence à partir des mesures sans rotation de l'antenne
Ho2f0 = sp.compute_SphericalWavesbasis_origin_to_field(XYZm, kvect, OSH, SH_center = np.array([0,0,0], dtype = float))
Cmn0  = sp.compute_SHcoefs(P, Ho2f0, O_SH_vect, lambda_reg=1e-4)

### Etape 2 - Construction de la base tournée via Wigner-D

Les harmoniques sphériques tournées par $R$ s'expriment comme :
$$\tilde{H}_n^m(\hat{r}) = b_n(kr_i)\sum_{m'=-n}^{n} D^n_{m'm}(R) \; Y_n^{m'}(\hat{r}) = \sum_{m'=-n}^{n} D^n_{m'm}(R) \; H_n^{m'}(\hat{r} )$$

En matriciel, la matrice de décomposition tournée est :
$$\tilde{\mathbf{H}} = \mathbf{H}_0 \; \mathbf{D}(R)$$
où $D(R)$ est la matrice bloc-diagonale de Wigner-D de taille $(N_{max}+1)^2 \times (N_{max}+1)^2$, avec un bloc $(2n+1) \times (2n+1)$
par ordre $n$.

In [8]:
from scipy.linalg import block_diag

ThR, PhR = np.radians(30), np.radians(45)  # Example rotation angles
R = get_rotation_matrix(ThR, PhR)

# Construct the full block-diagonal Wigner-D matrix for all orders 0 to OSH
DR_blocks = []
for l in range(OSH + 1):
    r_obj = Rotation.from_matrix(R)
    alp, bet, gam = r_obj.as_euler('zyz')
    dl = utils.wigner_D_matrix(l, alp, bet, gam)
    DR_blocks.append(dl)

# Combine into one large block diagonal matrix
DR = block_diag(*DR_blocks)
print(f"Ho2f0 shape: {Ho2f0.shape}")
print(f"DR shape: {DR.shape}")

Ho2fR = np.einsum('msf,st->mtf', Ho2f0, DR)


Ho2f0 shape: (256, 64, 2499)
DR shape: (64, 64)


### Etape 3 - Décomposition dans la base tournée
On décompose le même champ $\mathbf{p}$ sur la base tournée :
$$\tilde{C} = \tilde{\mathbf{H}}^+ \, \mathbf{p} = \left(\mathbf{H}_0 \, \mathbf{D}\right)^+ \mathbf{p}$$
Ces coefficients $\tilde{C}$ sont exprimés dans la base tournée.

In [9]:
CmnR  = sp.compute_SHcoefs(P, Ho2fR, O_SH_vect, lambda_reg=1e-4)

## Étape 4 — Retour dans la base d'origine via Wigner-D

Pour comparer avec $C_0$​, on ramène les coefficients dans la base non tournée :

$$C_{\text{back}} = \mathbf{D}(R) \; \tilde{C}$$

ordre par ordre :
$$C_{\text{back},n}^{m} = \sum_{m'=-n}^{n} D^n_{mm'}(R) \; \tilde{C}_n^{m'}$$

En théorie parfaite, $C_{\text{back}} = C_0$ exactement.

In [10]:
Cmnbk = DR @ CmnR

## Étape 5 — Mesure de l'erreur
L'écart entre $C_{\text{back}}$​ et $C_0$​ mesure la non-invariance numérique de la décomposition par rotation de la base :

$$\varepsilon = \frac{\| C_{\text{back}} - C_0 \|^2}{\| C_0 \|^2}$$
et pour chaque ordre:

$$\varepsilon_n = \frac{\sum_{m} |C_{\text{back},n}^m - C_{0,n}^m|^2}{\sum_{m} |C_{0,n}^m|^2}$$


In [11]:
# Calcul de l'erreur globale (sur tous les ordres et toutes les fréquences)
diff = Cmnbk - Cmn0
num_global = np.sum(np.abs(diff)**2)
den_global = np.sum(np.abs(Cmn0)**2)
epsilon = num_global / den_global

print(f"Erreur globale (epsilon) : {epsilon:.4e}")

# Calcul de l'erreur par ordre n
epsilon_n = np.zeros(OSH + 1)

print("\nErreur par ordre :")
for n in range(OSH + 1):
    # Les indices des harmoniques sphériques pour l'ordre n vont de n^2 à (n+1)^2 - 1
    idx_start = n**2
    idx_end = (n + 1)**2
    
    # Extraction des coefficients pour l'ordre n
    # Cmn a la forme (NbSH, Nbf)
    C0_n = Cmn0[idx_start:idx_end, :]
    Cbk_n = Cmnbk[idx_start:idx_end, :]
    
    # Calcul de l'erreur normalisée pour cet ordre
    diff_n = Cbk_n - C0_n
    num_n = np.sum(np.abs(diff_n)**2)
    den_n = np.sum(np.abs(C0_n)**2)
    
    if den_n > 0:
        epsilon_n[n] = num_n / den_n
    else:
        epsilon_n[n] = 0.0
        
    print(f"  Ordre {n} : {epsilon_n[n]:.4e}")

Erreur globale (epsilon) : 2.0715e-01

Erreur par ordre :
  Ordre 0 : 1.5368e-01
  Ordre 1 : 1.6575e-01
  Ordre 2 : 1.7986e-01
  Ordre 3 : 2.2956e-01
  Ordre 4 : 2.7648e-01
  Ordre 5 : 2.0048e-01
  Ordre 6 : 2.1155e-01
  Ordre 7 : 1.8197e-01


## Étape 6 — Caractérisation de l'erreur
Pour caractériser cette erreur de manière robuste, il faut évaluer si elle est stable ou dépend de la rotation choisie.

L'approche standard consiste à effectuer une simulation de Monte-Carlo sur un ensemble de rotations aléatoires (Step 6). Cela permet de vérifier si l'erreur reste négligeable quelle que soit l'orientation (θ,φ).

Workflow pour mettre en œuvre cette caractérisation statistique :
On génère N rotations aléatoires.
Pour chacune, on calcule l'erreur globale et l'erreur par ordre.
On trace les distributions (boxplots) pour visualiser la stabilité numérique.

Génération de rotations aléatoires : 
- On tire des angles (θ,φ) aléatoires uniformément sur la sphère.
- Calcul de la chaîne complète pour chaque rotation :
* Rotation de la base d'harmoniques sphériques.
* Projection du champ de pression mesuré sur cette nouvelle base.
* Retour dans la base d'origine via la matrice de Wigner-D inverse.
* Calcul de l'erreur : On compare les coefficients obtenus avec ceux de référence (sans rotation).
* Visualisation : Un boxplot (échelle logarithmique) montre la distribution de l'erreur par ordre.

Cela permettra de visualiser la stabilité numérique de votre décomposition SH. 

- Si l'erreur est faible ($<1e-10$  ou proche du bruit numérique), la méthode est robuste. 
- Si elle augmente avec l'ordre, cela peut indiquer des problèmes de conditionnement ou d'échantillonnage spatial.


In [ ]:
# --- Paramètres de la simulation ---
N_iter_max = 1000  # Nombre maximum d'itérations
Batch_size = 50   # Taille du lot pour le calcul parallèle (Increased)
Convergence_Threshold = 1e-4

NbViolToProcess = 6

# Cast global large arrays to single precision
# Ho2f0 is complex, so complex64. Pm is complex, so complex64.
Ho2f0 = Ho2f0.astype(np.complex64)

# Storage for each violin
violins_state = []
for v_idx in range(NbViolToProcess):
    state = {
        'convergence_errors': [],
        # Initialize with complex64 zeros
        'Cmn_sum': np.zeros_like(Cmn0, dtype=np.complex64), 
        'Ref_C_prev': None,
        'stop_met': False,
        # Cast Pressure to complex64
        'P': Pm[v_idx, Band, :].T.astype(np.complex64), 
        'total_iter': 0
    }
    violins_state.append(state)

# --- Define Grid for plots (angles) ---
n_th = 100
n_ph = 200
th_vec = np.linspace(0, np.pi, n_th)
ph_vec = np.linspace(0, 2*np.pi, n_ph)
TH, PH = np.meshgrid(th_vec, ph_vec)
angles_look = np.column_stack((TH.flatten(), PH.flatten())) 

print(f"Lancement de la simulation pour {NbViolToProcess} violons en parallèle (Rotations indépendantes)...")

import plotly.graph_objects as go
from scipy.spatial.transform import Rotation
from joblib import Parallel, delayed
from scipy.linalg import block_diag
import random

# --- Figure 1: Convergence (All Violins on one plot) ---
fig1 = go.FigureWidget()
fig1.update_layout(title_text="Convergence (Relative Difference)",
                   yaxis_type="log", yaxis_title="Rel. Diff", xaxis_title="Iterations",
                   width=1000, height=400)
for i in range(NbViolToProcess):
    fig1.add_trace(go.Scatter(x=[], y=[], mode='lines', name=f'V{i+1}'))


# --- Figure 2: Concatenated Heatmaps ---
fig2 = go.FigureWidget()
fig2.update_layout(title_text="Directivity Patterns (V1 ... V6)",
                   width=1000, height=300,
                   yaxis=dict(scaleanchor="x", scaleratio=1))
# Start with empty large matrix
fig2.add_trace(go.Heatmap(z=[[0]], colorscale='icefire', showscale=True))

display(fig1)
display(fig2)

# --- Processing Function ---
# Now generates a DIFFERENT rotation for EACH violin
def process_single_step_indep_rotations(seed, Ps_list):
    # Seed generator for this job
    rng = np.random.RandomState(seed)
    
    results = []
    
    for P_v in Ps_list:
        # 1. Unique Rotation for this violin
        th_rand = np.arccos(2 * rng.rand() - 1) 
        ph_rand = 2 * np.pi * rng.rand()        
        R_i = get_rotation_matrix(ph_rand, th_rand)
        
        # 2. Wigner-D for this rotation
        # Optimization: Reuse wigner_D_matrix if possible, but angles differ
        DR_blocks_i = [
            utils.wigner_D_matrix(l, *Rotation.from_matrix(R_i).as_euler('zyz'))
            for l in range(OSH + 1)
        ]
        # Ensure Wigner-D is complex64
        DR_i = block_diag(*DR_blocks_i).astype(np.complex64)
        
        # 3. Apply to Basis
        # Ho2f0 is global (now complex64). Ho2fR_i depends on DR_i
        Ho2fR_i = np.matmul(Ho2f0.transpose(0, 2, 1), DR_i).transpose(0, 2, 1)

        # 4. Compute Coefs
        # P_v is already complex64. Ho2fR_i is complex64.
        CmnR_i = sp.compute_SHcoefs(P_v, Ho2fR_i, O_SH_vect, lambda_reg=1e-4)
        
        # Ensure CmnR_i is complex64 (compute_SHcoefs might return double)
        CmnR_i = CmnR_i.astype(np.complex64)
        
        Cmnbk_i = DR_i @ CmnR_i
        results.append(Cmnbk_i)
        
    return results # List [Cmnbk_v1(Rot1), Cmnbk_v2(Rot2), ...]

# --- Main Loop ---
global_iter = 0
all_stopped = False
# Ps_all elements are already cast to complex64 in violins_state
Ps_all = [st['P'] for st in violins_state]

while global_iter < N_iter_max and not all_stopped:
    current_batch_size = min(Batch_size, N_iter_max - global_iter)
    seeds = [random.randint(0, 1000000) for _ in range(current_batch_size)]
    
    # Run batch - Use all available cores (-1)
    results_batch = Parallel(n_jobs=-1)(delayed(process_single_step_indep_rotations)(s, Ps_all) for s in seeds)
    
    for res_multi in results_batch:
        global_iter += 1
        for v_idx in range(NbViolToProcess):
            st = violins_state[v_idx]
            st['total_iter'] += 1
            Cmnbk_i = res_multi[v_idx]
            st['Cmn_sum'] += Cmnbk_i
            Ref_C_current = st['Cmn_sum'] / st['total_iter']
            
            # Conv
            if st['Ref_C_prev'] is not None:
                norm_prev = np.linalg.norm(st['Ref_C_prev'])
                conv_metric = (np.linalg.norm(Ref_C_current - st['Ref_C_prev']) / norm_prev) if norm_prev > 1e-15 else 0.0
                st['convergence_errors'].append(conv_metric)
                if conv_metric < Convergence_Threshold: st['stop_met'] = True
            else:
                st['convergence_errors'].append(1.0)
            st['Ref_C_prev'] = np.copy(Ref_C_current)    

    if all(st['stop_met'] for st in violins_state): all_stopped = True
        
    # --- Batch Plot Update ---
    current_x = np.arange(1, global_iter + 1)
    
    # 1. Update Convergence
    with fig1.batch_update():
        for v_idx in range(NbViolToProcess):
            fig1.data[v_idx].x = current_x
            fig1.data[v_idx].y = violins_state[v_idx]['convergence_errors']
    # 2. Update Heatmap (Concatenated)
    # Collect all maps
    maps = []
    for v_idx in range(NbViolToProcess):
        st = violins_state[v_idx]
        Ref_C_curr = st['Cmn_sum'] / st['total_iter']
        Ref_Diag = sp.compute_Dinf_from_SH_coefs_at_origin(Ref_C_curr, angles_look, kvect)
        if Ref_Diag.ndim == 3: hm_data = np.real(np.sum(Ref_Diag, axis=2))
        elif Ref_Diag.ndim == 2: hm_data = np.real(np.sum(Ref_Diag, axis=1)).reshape(n_ph, n_th)
        else: hm_data = np.zeros((n_ph, n_th))
        maps.append(hm_data)
    
    full_map = np.concatenate(maps, axis=1)
    
    mx = np.max(np.abs(full_map))
    with fig2.batch_update():
        fig2.data[0].z = full_map
        fig2.data[0].zmin = -mx
        fig2.data[0].zmax = mx

print(f"Terminé. Itérations: {global_iter}")

# --- Save Final Results ---
print("Saving final Ref_C and Ref_Diag...")
Ref_C_all = []
Ref_Diag_all = []

for v_idx in range(NbViolToProcess):
    st = violins_state[v_idx]
    # Recompute final average Cmn
    Ref_C_final = st['Cmn_sum'] / st['total_iter']
    # Recompute final Directivity
    Ref_Diag_final = sp.compute_Dinf_from_SH_coefs_at_origin(Ref_C_final, angles_look, kvect)
    
    Ref_C_all.append(Ref_C_final)
    Ref_Diag_all.append(Ref_Diag_final)

# Convert to arrays
Ref_C_all = np.array(Ref_C_all)
Ref_Diag_all = np.array(Ref_Diag_all)

save_path = './../results/Optimized_Cmn_and_Diag.npz'
np.savez(save_path, Ref_C=Ref_C_all, Ref_Diag=Ref_Diag_all, angles_look=angles_look, kvect=kvect)
print(f"Saved to {save_path}")

Lancement de la simulation pour 6 violons en parallèle (Rotations indépendantes)...


FigureWidget({
    'data': [{'mode': 'lines',
              'name': 'V1',
              'type': 'scatter',
              'uid': '24b7796c-6ac0-4f04-99ff-9cc12e5b2636',
              'x': [],
              'y': []},
             {'mode': 'lines',
              'name': 'V2',
              'type': 'scatter',
              'uid': '019f3d9f-6bcf-427e-9e55-ef35ccff4ff2',
              'x': [],
              'y': []},
             {'mode': 'lines',
              'name': 'V3',
              'type': 'scatter',
              'uid': '3a1bde38-1f16-4363-921a-0d739a453c84',
              'x': [],
              'y': []},
             {'mode': 'lines',
              'name': 'V4',
              'type': 'scatter',
              'uid': '6ec2bc2f-eadf-498a-a8c3-b661041afdf0',
              'x': [],
              'y': []},
             {'mode': 'lines',
              'name': 'V5',
              'type': 'scatter',
              'uid': '606476af-b25e-40af-a447-ede769257372',
              'x': [],
         

FigureWidget({
    'data': [{'colorscale': [[0.0, '#000000'], [0.0625, '#001f4d'], [0.125,
                             '#003786'], [0.1875, '#0e58a8'], [0.25, '#217eb8'],
                             [0.3125, '#30a4ca'], [0.375, '#54c8df'], [0.4375,
                             '#9be4ef'], [0.5, '#e1e9d1'], [0.5625, '#f3d573'],
                             [0.625, '#e7b000'], [0.6875, '#da8200'], [0.75,
                             '#c65400'], [0.8125, '#ac2301'], [0.875, '#820000'],
                             [0.9375, '#4c0000'], [1.0, '#000000']],
              'showscale': True,
              'type': 'heatmap',
              'uid': 'e47d2626-3394-48b5-8cbf-b92efb6ee0d9',
              'z': [[0]]}],
    'layout': {'height': 300,
               'template': '...',
               'title': {'text': 'Directivity Patterns (V1 ... V6)'},
               'width': 1000,
               'yaxis': {'scaleanchor': 'x', 'scaleratio': 1}}
})